# Simple Pad — HFSS Eigenmode and EPR

Public synthetic tutorial candidate `orpen-simple-pad-public-tutorial`
under `scgsim-docs-1.0.2-20261006`: semantic state CONVERGING, delivery
LOCAL CANDIDATE, no test writes. Owner: OrPen SC PDK. Canonical layout
authority is `square_pad_capacitor`; reusable runtime belongs to SCGSim.
The authored pad and connected surrounding Ground are finite Al (0–0.2
um), above one 500 um Si die (-500–0 um). There is no second die or
fabricated inter-die gap. The source locator sheet spans
x=99.999–120.001, y=-1–1 um: 20.002 by 2 um, centered at (110, 0),
directed +X. It overlaps each terminal by one source DBU (0.001 um) for
exact contact binding; physical metal and the 20 um gap are unchanged.
The component owns this overlap metadata.

Historical source receipt: OrPen base `36b6e6b`; SCGSim runtime 1.0.1
source `9c032e398f2ebabc2dc53965b2bde5c346a061bc`. This identifies the
earlier authoring cohort, not the revised v2 source below. Current
source-only SCGSim v2 candidate targets public revision
`cc1424c84ffe9d295a57dbabb3c2ab6431110f6c` (`2.0.0.dev1`). Use a
separate consumer environment; the ordinary OrPen project still pins
`2eed1eb`. For example, in a fresh virtual environment with this OrPen
checkout installed:

``` bash
python -m pip install "scgsim[palace,visualization,aedt] @ git+https://github.com/OrPenStrike/scgsim.git@cc1424c84ffe9d295a57dbabb3c2ab6431110f6c"
```

Do not use the ordinary project’s `uv sync` as a v2 installation
instruction. This alignment was not executed: preparation, native
modeling, mesh, solve, fields, Resolve and Report remain unobserved for
these revised sources. Historical results retain their original runtime
and run identity. Quarto execution is disabled. This checkpoint has no
native geometry, mesh, solve, capacitance, frequency, EPR or adequacy
claim. Run IDs below identify new preparations, not pre-existing
evidence; change the ID for each new run. Native search/convergence
controls are inputs, not research adequacy Gates.

The authored east junction sheet is non-galvanic, directed +X from pad
to Ground, L=10 nH and C=0 (an explicit inductive-only model). Surface
EPR uses OrPen LiteratureCentral synthetic assumptions, not measured
loss. HFSS keeps its native outer default: no new Region
PerfectE/Radiation/PML assignment is made, and the unobserved implicit
boundary type is not asserted. Palace Eigenmode’s natural PMC convention
differs; this is not an equivalence claim. Native HFSS construction and
saved-field interpretation require the actual licensed AEDT host.
Adaptive EPR and later saved-field analysis are distinct; this tutorial
requests adaptive surface/bulk/junction evidence only.

## Setup and Imports

In [ ]:
from pathlib import Path
import os
import subprocess
import importlib.metadata

import gdsfactory as gf
from IPython.display import display
from scgsim.aedt import (
    AedtResources, EigenmodeSim, EigenmodeRunControl, EprAnalysisRequest,
    SurfaceEprSpec, planar_junction_from_port, prepare_planar_geometry_input,
    resolve_results,
)
from scgsim.geometry import GeometryPlan
import orpen_sc_pdk
from orpen_sc_pdk import get_material_records, get_interface_preset_records, validate_interface_preset_records
from orpen_sc_pdk.tech import LAYER, get_single_die_layer_stack

## Setup and Run Controls

In [ ]:
WORKFLOW_ACTION = "prepare_handoff"  # prepare_handoff | run | analyze_handoff
RUN_ID = "simple_pad_aedt_hfss_eigenmode_v2_20261009_01"
OUTPUT_ROOT = Path.cwd() / ".artifacts" / "SimplePad"
RUN_DIR = OUTPUT_ROOT / RUN_ID
PAD_SIDE_UM = 200.0
GAP_UM = 20.0
DIE_SIDE_UM = 1000.0
JUNCTION_WIDTH_UM = 2.0
VACUUM_PADDING_UM = (250.0, 250.0, 1000.0)
AEDT_VERSION = "2024.2"  # Current SCGSim Eigenmode facade binds this runtime version.
PYAEDT_VERSION = "1.3.0"
SCGSIM_RUNTIME_REVISION = "cc1424c84ffe9d295a57dbabb3c2ab6431110f6c"
SETUP_NAME = "Setup1"
MAXIMUM_PASSES = 10
CONVERGENCE_PERCENT = 1.0
RESOURCES = AedtResources(cores=4, ram_limit_percent=80)
ROUTE = "B"
SEARCH_FREQUENCY_GHZ = 1.0
NUM_MODES = 1
PORT_NAME = "chip/o_junction_lumped"
JUNCTION_ID = "jj"
PORT_INDUCTANCE_H = 10e-9
PORT_CAPACITANCE_F = 0.0
EPR_MARGINS_UM = (0.0,)
PRESET_NAMES = {kind: f"LiteratureCentral_{kind}" for kind in ("MA", "MS", "SA")}
if WORKFLOW_ACTION not in {"prepare_handoff", "run", "analyze_handoff"}:
    raise ValueError("Unknown WORKFLOW_ACTION")

## Create Simulation Component / Coupon

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    orpen_sc_pdk.activate()
    gf.clear_cache()
    component = gf.get_component(
        "square_pad_capacitor", pad_side_um=PAD_SIDE_UM, gap_um=GAP_UM,
        die_side_um=DIE_SIDE_UM, junction_width_um=JUNCTION_WIDTH_UM,
    )
    layer_stack = get_single_die_layer_stack()
    material_records = get_material_records()
    plan = GeometryPlan(
        component=component, root_id="chip", layer_stack=layer_stack,
        material_records=material_records, coupon_padding_um=0.0,
    )
    plan.set_nets({"pad": ("chip/PAD",), "ground": ("chip/GROUND",)})
    plan.set_vacuum_region(padding=VACUUM_PADDING_UM)
    snapshot = plan.prepare()
    display(component)

## Initialize AEDT Project / App

Preparation creates a portable SCGSim package, not a live Desktop
session. Only the generated runner in the explicit `run` action owns
native AEDT.

In [ ]:
display({
    "requested_aedt": AEDT_VERSION, "requested_pyaedt": PYAEDT_VERSION,
    "installed_pyaedt": importlib.metadata.version("pyaedt"),
    "installed_scgsim": importlib.metadata.version("scgsim"),
    "runtime_source": SCGSIM_RUNTIME_REVISION, "run_id": RUN_ID,
})

## Import GDS and Build the HFSS/Q3D/Q2D Model

The shared runtime exports the canonical source GDS and normalizes it in
the snapshot. Bind the real authored port through the public helper,
then prepare the same junction-aware catalogue used by the final
handoff. Neither operation opens AEDT. There is no notebook-local parser
or source-normalization clone.

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    junction = planar_junction_from_port(
        snapshot.geometry_input, port_name=PORT_NAME, junction_id=JUNCTION_ID,
        terminal_a_net="pad", terminal_b_net="ground",
        inductance_h=PORT_INDUCTANCE_H, capacitance_f=PORT_CAPACITANCE_F,
    )
    prepared = prepare_planar_geometry_input(
        snapshot.geometry_input, prepared_stack=snapshot.stack,
        route=ROUTE, junctions=(junction,),
    )
    catalogue = prepared.contribution_catalog

## Geometry Verification

This is source inspection, not native geometry readback. The actual
runner must retain native material, object, net, bounds and assignment
observations in its receipt; analyze those observations after retrieval.

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    display({"component_bbox_um": str(component.dbbox()), "ports": component.ports})
    display(catalogue)

## Materials and Boundaries

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    presets = validate_interface_preset_records(get_interface_preset_records())
    surface_specs = []
    for record in catalogue:
        kind = record["interface_kind"]
        if kind not in PRESET_NAMES:
            continue
        preset_name = PRESET_NAMES[kind]
        preset = presets[preset_name]
        surface_specs.append(SurfaceEprSpec(
            contribution_id=record["contribution_id"],
            source_polygon_id=record["source_polygon_ids"][0],
            interface_kind=kind, field_side=record["field_side"],
            margins_um=EPR_MARGINS_UM,
            film_thickness_m=preset["thickness"] * 1e-6,
            film_relative_permittivity=preset["permittivity"],
            loss_tangent=preset["loss_tangent"], source=preset["source"], preset=preset_name,
        ))
    sim = EigenmodeSim()
    sim.set_plan(snapshot)
    sim.set_output_dir(RUN_DIR)
    sim.set_surface_epr(contributions=surface_specs)

## Ports / Nets / Excitations

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    sim.add_junction(junction)
    sim.set_epr_request(EprAnalysisRequest(
        mode_indices=(1,),
        surface_contribution_ids=tuple(item.contribution_id for item in surface_specs),
        junction_ids=(JUNCTION_ID,),
        bulk_domain_ids=None,  # All configured native bulk domains, including Region.
    ))

## Simulation Setup

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    run_control = EigenmodeRunControl(
        setup_name=SETUP_NAME, minimum_frequency_ghz=SEARCH_FREQUENCY_GHZ,
        num_modes=NUM_MODES, maximum_passes=MAXIMUM_PASSES,
        maximum_delta_frequency_percent=CONVERGENCE_PERCENT,
    )
    sim.set_eigenmode(project_name=RUN_ID, design_name="SimplePadEigenmode",
                      route=ROUTE, run_control=run_control, resources=RESOURCES)

## Simulation Configuration

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    display(sim.input_summary())

## Solve and Export

A fresh handoff directory is required by SCGSim. Preparation never opens
AEDT or solves. Change the action to `run` only for explicitly
authorized native host execution; that branch invokes only the generated
runner.

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    HANDOFF = sim.prepare_handoff()
    display(HANDOFF)
    if WORKFLOW_ACTION == "run":
        subprocess.run([str(HANDOFF.script_path)], cwd=HANDOFF.run_dir, check=True)

## Adaptive-Pass Convergence / Solver Diagnostics

Use a fresh kernel and an explicit returned run directory, not live
preparation state. The strict resolver reads sealed receipt identity,
copied inputs and native readback. Missing or corrupt artifacts fail;
nothing synthesizes a successful return. Retain the exact
preparation-side spec/manifest identity independently when transporting
the package.

In [ ]:
if WORKFLOW_ACTION == "analyze_handoff":
    RETURNED_RUN_DIR = Path(os.environ["SIMPLE_PAD_RETURNED_RUN_DIR"])
    result = resolve_results(RETURNED_RUN_DIR)
    # Trust/identity and native numerical/performance evidence precede physics.
    result.show_all_results(show_details=True)

## Results: Plots and Readable Tables

### Physics Analysis Results

In [ ]:
if WORKFLOW_ACTION == "analyze_handoff":
    display(result.physics_results())
    display(result.epr_result())

### Simulation Performance / Benchmarks

In [ ]:
if WORKFLOW_ACTION == "analyze_handoff":
    display(result.simulation_benchmark())

## Save and Release AEDT

The generated runner saves the exact project and releases only its owned
Desktop session. The strict resolver requires successful save/release
and hash-bound output. No native outcome exists at this source
checkpoint.

In [ ]:
if WORKFLOW_ACTION == "analyze_handoff":
    display({"project": result.project_path, "receipt": result.receipt_path})
elif WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    display(HANDOFF.archive_path)